In [1]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0,
)

texts = [
    "I loved this movie, the acting was brilliant.",
    "The subtitles were out of sync and the audio was terrible.",
]

print(classifier(texts))

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998792409896851}, {'label': 'NEGATIVE', 'score': 0.9997938275337219}]


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name).to("cuda")

text = "I loved this movie, the acting was brilliant."

tokens = tokenizer.tokenize(text)
print(tokens)

inputs = tokenizer(text, return_tensors="pt")
print(inputs["input_ids"])

with torch.no_grad():
    out = model(**inputs.to("cuda"))

print(out.logits)
probs = torch.softmax(out.logits, dim=-1)
print(probs)
print(model.config.id2label)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

['i', 'loved', 'this', 'movie', ',', 'the', 'acting', 'was', 'brilliant', '.']
tensor([[ 101, 1045, 3866, 2023, 3185, 1010, 1996, 3772, 2001, 8235, 1012,  102]])
tensor([[-4.3403,  4.6818]], device='cuda:0')
tensor([[1.2070e-04, 9.9988e-01]], device='cuda:0')
{0: 'NEGATIVE', 1: 'POSITIVE'}


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(name, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(name).to("cuda")

lines = [
    "Where are you going?",
    "I loved this movie, the acting was brilliant.",
    "The meeting has been moved to tomorrow morning.",
]

inputs = tokenizer(lines, return_tensors="pt", padding=True).to("cuda")
tamil_id = tokenizer.convert_tokens_to_ids("tam_Taml")

with torch.no_grad():
    generated = model.generate(
        **inputs,
        forced_bos_token_id=tamil_id,
        max_length=100,
    )

outputs = tokenizer.batch_decode(generated, skip_special_tokens=True)

for en, ta in zip(lines, outputs):
    print(en)
    print(ta)
    print()

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Where are you going?
எங்கே போகிறாய்?

I loved this movie, the acting was brilliant.
நான் இந்த படத்தை நேசித்தேன், நடிப்பு பிரகாசமாக இருந்தது.

The meeting has been moved to tomorrow morning.
கூட்டம் நாளை காலைக்கு ஒத்திவைக்கப்பட்டுள்ளது.



In [4]:
!pip install -q sacrebleu

import sacrebleu

refs = [
    "WRITE NATURAL TAMIL FOR: Where are you going?",
    "WRITE NATURAL TAMIL FOR: I loved this movie, the acting was brilliant.",
    "WRITE NATURAL TAMIL FOR: The meeting has been moved to tomorrow morning.",
]

chrf = sacrebleu.corpus_chrf(outputs, [refs])
bleu = sacrebleu.corpus_bleu(outputs, [refs])

print("chrF:", round(chrf.score, 1))
print("BLEU:", round(bleu.score, 1))

for hyp, ref in zip(outputs, refs):
    print(round(sacrebleu.sentence_chrf(hyp, [ref]).score, 1), "|", hyp, "|", ref)

chrF: 0.5
BLEU: 1.1
0.5 | எங்கே போகிறாய்? | WRITE NATURAL TAMIL FOR: Where are you going?
0.6 | நான் இந்த படத்தை நேசித்தேன், நடிப்பு பிரகாசமாக இருந்தது. | WRITE NATURAL TAMIL FOR: I loved this movie, the acting was brilliant.
0.3 | கூட்டம் நாளை காலைக்கு ஒத்திவைக்கப்பட்டுள்ளது. | WRITE NATURAL TAMIL FOR: The meeting has been moved to tomorrow morning.


In [5]:
import sacrebleu

refs = [
    "நீங்கள் எங்கே போகிறீர்கள்?",
    "இந்தப் படம் எனக்கு ரொம்பப் பிடித்திருந்தது, நடிப்பு அருமையாக இருந்தது.",
    "கூட்டம் நாளை காலைக்கு மாற்றப்பட்டுள்ளது.",
]

print("sanity check (should be 100):", round(sacrebleu.corpus_chrf(refs, [refs]).score, 1))

chrf = sacrebleu.corpus_chrf(outputs, [refs])
bleu = sacrebleu.corpus_bleu(outputs, [refs])
print("chrF:", round(chrf.score, 1))
print("BLEU:", round(bleu.score, 1))

for hyp, ref in zip(outputs, refs):
    print(round(sacrebleu.sentence_chrf(hyp, [ref]).score, 1), "|", hyp, "|", ref)

sanity check (should be 100): 100.0
chrF: 49.3
BLEU: 15.6
40.3 | எங்கே போகிறாய்? | நீங்கள் எங்கே போகிறீர்கள்?
36.1 | நான் இந்த படத்தை நேசித்தேன், நடிப்பு பிரகாசமாக இருந்தது. | இந்தப் படம் எனக்கு ரொம்பப் பிடித்திருந்தது, நடிப்பு அருமையாக இருந்தது.
76.2 | கூட்டம் நாளை காலைக்கு ஒத்திவைக்கப்பட்டுள்ளது. | கூட்டம் நாளை காலைக்கு மாற்றப்பட்டுள்ளது.
